In [1]:
import torch
import torch.nn
import os
import codecs
import json
import csv
from torchtext.data.functional import generate_sp_model
import torchtext.transforms as T
import torch.utils.data as D
from torch import nn, optim
import lightning as L
from collections import OrderedDict
from torchtext.vocab import vocab
import math
import numpy as np
from lightning.pytorch.loggers import WandbLogger

In [2]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [3]:
corpus_name = "movie-corpus"
corpus = os.path.join("data", corpus_name)

def printLines(file, n=10):
    with open(file, 'rb') as datafile:
        lines = datafile.readlines()
    for line in lines[:n]:
        print(line)

printLines(os.path.join(corpus, "utterances.jsonl"))


b'{"id": "L1045", "conversation_id": "L1044", "text": "They do not!", "speaker": "u0", "meta": {"movie_id": "m0", "parsed": [{"rt": 1, "toks": [{"tok": "They", "tag": "PRP", "dep": "nsubj", "up": 1, "dn": []}, {"tok": "do", "tag": "VBP", "dep": "ROOT", "dn": [0, 2, 3]}, {"tok": "not", "tag": "RB", "dep": "neg", "up": 1, "dn": []}, {"tok": "!", "tag": ".", "dep": "punct", "up": 1, "dn": []}]}]}, "reply-to": "L1044", "timestamp": null, "vectors": []}\n'
b'{"id": "L1044", "conversation_id": "L1044", "text": "They do to!", "speaker": "u2", "meta": {"movie_id": "m0", "parsed": [{"rt": 1, "toks": [{"tok": "They", "tag": "PRP", "dep": "nsubj", "up": 1, "dn": []}, {"tok": "do", "tag": "VBP", "dep": "ROOT", "dn": [0, 2, 3]}, {"tok": "to", "tag": "TO", "dep": "dobj", "up": 1, "dn": []}, {"tok": "!", "tag": ".", "dep": "punct", "up": 1, "dn": []}]}]}, "reply-to": null, "timestamp": null, "vectors": []}\n'
b'{"id": "L985", "conversation_id": "L984", "text": "I hope so.", "speaker": "u0", "meta": {

In [5]:
# Splits each line of the file to create lines and conversations
def loadLinesAndConversations(fileName):
    lines = {}
    conversations = {}
    with open(fileName, 'r', encoding='iso-8859-1') as f:
        for line in f:
            lineJson = json.loads(line)
            # Extract fields for line object
            lineObj = {}
            lineObj["lineID"] = lineJson["id"]
            lineObj["characterID"] = lineJson["speaker"]
            lineObj["text"] = lineJson["text"]
            lines[lineObj['lineID']] = lineObj

            # Extract fields for conversation object
            if lineJson["conversation_id"] not in conversations:
                convObj = {}
                convObj["conversationID"] = lineJson["conversation_id"]
                convObj["movieID"] = lineJson["meta"]["movie_id"]
                convObj["lines"] = [lineObj]
            else:
                convObj = conversations[lineJson["conversation_id"]]
                convObj["lines"].insert(0, lineObj)
            conversations[convObj["conversationID"]] = convObj

    return lines, conversations


# Extracts pairs of sentences from conversations
def extractSentencePairs(conversations):
    qa_pairs = []
    for conversation in conversations.values():
        # Iterate over all the lines of the conversation
        for i in range(len(conversation["lines"]) - 1):  # We ignore the last line (no answer for it)
            inputLine = conversation["lines"][i]["text"].strip()
            targetLine = conversation["lines"][i+1]["text"].strip()
            # Filter wrong samples (if one of the lists is empty)
            if inputLine and targetLine:
                qa_pairs.append([inputLine, targetLine])
    return qa_pairs

In [6]:
# Define path to new file
datafile = os.path.join(corpus, "formatted_movie_lines.txt")

delimiter = '\t'
# Unescape the delimiter
delimiter = str(codecs.decode(delimiter, "unicode_escape"))

# Initialize lines dict and conversations dict
lines = {}
conversations = {}
# Load lines and conversations
print("\nProcessing corpus into lines and conversations...")
lines, conversations = loadLinesAndConversations(os.path.join(corpus, "utterances.jsonl"))

# Write new csv file
print("\nWriting newly formatted file...")
with open(datafile, 'w', encoding='utf-8') as outputfile:
    writer = csv.writer(outputfile, delimiter=delimiter, lineterminator='\n')
    for pair in extractSentencePairs(conversations):
        writer.writerow(pair)

# Print a sample of lines
print("\nSample lines from file:")
printLines(datafile)


Processing corpus into lines and conversations...

Writing newly formatted file...

Sample lines from file:
b'They do to!\tThey do not!\n'
b'She okay?\tI hope so.\n'
b"Wow\tLet's go.\n"
b'"I\'m kidding.  You know how sometimes you just become this ""persona""?  And you don\'t know how to quit?"\tNo\n'
b"No\tOkay -- you're gonna need to learn how to lie.\n"
b"I figured you'd get to the good stuff eventually.\tWhat good stuff?\n"
b'What good stuff?\t"The ""real you""."\n'
b'"The ""real you""."\tLike my fear of wearing pastels?\n'
b'do you listen to this crap?\tWhat crap?\n'
b"What crap?\tMe.  This endless ...blonde babble. I'm like, boring myself.\n"


In [6]:
from torchtext.data.functional import generate_sp_model

num_words = 10000
generate_sp_model(datafile, vocab_size=num_words)

sentencepiece_trainer.cc(177) LOG(INFO) Running command: --input=data/movie-corpus/formatted_movie_lines.txt --model_prefix=m_user --vocab_size=10000 --model_type=unigram
sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: data/movie-corpus/formatted_movie_lines.txt
  input_format: 
  model_prefix: m_user
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  treat_whitespace_as_suffix: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos

In [26]:
padding_idx = 0
bos_idx = 1
eos_idx = 2

od = OrderedDict()

with open('m_user.vocab', 'r') as file:
    for idx, line in enumerate(file):
        word = line.split('\t')[0]
        od[word] = idx

v = vocab(od, specials=['<unk>'])
v.set_default_index(0)

In [11]:
max_seq_len = 50

text_transform = T.Sequential(
    T.SentencePieceTokenizer('m_user.model'),
    T.VocabTransform(v),
    T.Truncate(max_seq_len - 2),
    T.AddToken(token=bos_idx, begin=True),
    T.AddToken(token=eos_idx, begin=False),
    T.ToTensor(),
    T.PadTransform(max_seq_len, padding_idx)
)

In [12]:
class StarWarsDataset(D.Dataset):
    def __init__(self, pairs, transform = None):
        self.pairs = pairs
        self.transform = transform
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        inp, tgt = self.pairs[idx]
        
        if self.transform:
            inp = self.transform(inp).to(device)
            tgt = self.transform(tgt).to(device)
        # Size is greater than max word so we can crop last pad to get same dimension
        inp = inp[:-1]
        tgt_y = tgt[1:]
        tgt = tgt[:-1]
        return inp, tgt, tgt_y

dataset = StarWarsDataset(extractSentencePairs(conversations), text_transform)
train_set, val_set = D.random_split(dataset, [0.9, 0.1])

In [14]:
batch_size = 64

train_loader = D.DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = D.DataLoader(val_set, batch_size=batch_size, shuffle=False)

In [13]:
class PositionalEncoding(nn.Module):
    "Implement the PE function."
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) *
                             -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [14]:
learning_rate = 1e-3
d_model = 512

class Seq2Seq(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(num_words, d_model)
        self.encoding = PositionalEncoding(d_model, 0.1)
        self.transformer = nn.Transformer()
        self.linear = nn.Linear(d_model, num_words)
        self.loss = nn.CrossEntropyLoss(reduction='none')
    
    def create_pad_mask(self, x):
        return x == padding_idx
    
    def forward(self, inp_seq, tgt_seq):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_seq.shape[1], device=device)
        inp_padding_mask = self.create_pad_mask(inp_seq)
        tgt_padding_mask = self.create_pad_mask(tgt_seq)
        inp_embed = self.embedding(inp_seq) * np.sqrt(d_model)
        tgt_embed = self.embedding(tgt_seq) * np.sqrt(d_model)
        inp_encoded = self.encoding(inp_embed)
        tgt_encoded = self.encoding(tgt_embed)
        inp_permuted = inp_encoded.permute(1,0,2)
        tgt_permuted = tgt_encoded.permute(1,0,2)
        output = self.transformer(inp_permuted, tgt_permuted, tgt_mask=tgt_mask, src_key_padding_mask=inp_padding_mask, tgt_key_padding_mask=tgt_padding_mask)
        output = self.linear(output)
        return output
    
    def training_step(self, batch, batch_idx):
        inp, tgt, tgt_y = batch
        output = self.forward(inp, tgt)
        output = output.permute(1,2,0)
        mask = self.create_pad_mask(tgt_y)
        loss = self.loss(output, tgt).masked_select(mask).mean()
        self.log('train_loss', loss)
        return loss
    
    def validation_step(self, batch, batch_idx):
        inp, tgt, tgt_y = batch
        output = self.forward(inp, tgt)
        output = output.permute(1,2,0)
        mask = self.create_pad_mask(tgt_y)
        loss = self.loss(output, tgt).masked_select(mask).mean()
        self.log('val_loss', loss)
        return loss
    
    def configure_optimizers(self):
        opt = optim.AdamW(self.parameters(), lr=learning_rate)
        return opt

model = Seq2Seq().to(device)

if os.path.isfile('chatbot.pt'):
    model.load_state_dict(torch.load('chatbot.pt'))

In [18]:
wandb_logger = WandbLogger(project="Chatbot")

trainer = L.Trainer(max_epochs=10, logger=wandb_logger)
trainer.fit(model=model, train_dataloaders=train_loader, val_dataloaders=val_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type               | Params
---------------------------------------------------
0 | embedding   | Embedding          | 5.1 M 
1 | encoding    | PositionalEncoding | 0     
2 | transformer | Transformer        | 44.1 M
3 | linear      | Linear             | 5.1 M 
4 | loss        | CrossEntropyLoss   | 0     
---------------------------------------------------
54.4 M    Trainable params
0         Non-trainable params
54.4 M    Total params
217.562   Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [19]:
torch.save(model.state_dict(), 'chatbot.pt')

In [40]:
model.eval()

with torch.no_grad():
    inp = input('>>>')
    inp = text_transform(inp).unsqueeze(0).to(device)
    tgt = torch.tensor([[bos_idx]], device=device)

    while True:
        output = model(inp, tgt)[:, :, 2:].argmax(-1)[[-1]] + 2
        tgt = torch.cat((tgt, output), dim=-1)
        
        word = v.lookup_token(output)
        print(word, end='')
        # output = np.argmax(output)
        # word = v.lookup_token(output)
        # tgt = torch.cat((tgt, torch.tensor(output, device=device).unsqueeze(0).unsqueeze(0)), dim=-1)
        # print(word, end='')
        if output == eos_idx or tgt.shape[1] > 20:
            break

>>> hello there


</s>